# Summarize · news · entity description

Tests `/summarize` with `profile="entity_desc"`.

The service joins `evidence` into `evidence_text`, then applies **TextRank extractive pre-reduction** before the LLM prompt when `evidence_text` exceeds `SUMMARIZE_MAX_INPUT_WORDS` (default 400 words).

Cases 4 and 5 include 11-12 evidence items totalling >400 words to exercise the extractive path.

Fixtures: `eval/fixtures/summarize-news-entity-desc.json`

In [ ]:
import os, json, requests
from pathlib import Path

NLP_BASE_URL = os.environ.get('NLP_BASE_URL', 'http://localhost:9907').rstrip('/')
ENDPOINT     = f'{NLP_BASE_URL}/summarize'
HEADERS      = {'Content-Type': 'application/json', 'X-API-Key': f'{os.environ.get("NLP_API_KEY")}'}
TIMEOUT      = 120

MAX_INPUT_WORDS = 400   # mirrors SUMMARIZE_MAX_INPUT_WORDS in the service

CASES = json.loads(Path('fixtures/summarize-news-entity-desc.json').read_text(encoding='utf-8'))
print(f'{len(CASES)} cases -> {ENDPOINT}')

In [ ]:
def call_summarize(inp):
    r = requests.post(ENDPOINT, json=inp, headers=HEADERS, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json().get('result', {})


def extractive_would_trigger(fields):
    """Replicate service logic: join evidence list, count words."""
    evidence = fields.get('evidence', [])
    evidence_text = "\n".join(f"{i+1}. {e}" for i, e in enumerate(evidence))
    wc = len(evidence_text.split())
    return wc > MAX_INPUT_WORDS, wc


def render(result, case):
    fields = case['input']['fields']
    checks = case.get('expected', {}).get('checks', {})

    triggered, wc = extractive_would_trigger(fields)
    flag = '\u26a1 extractive' if triggered else '         '
    print(f'  {flag}  evidence_text: {wc} words  ({len(fields.get("evidence", []))} items)')
    print(f'  {fields.get("type", "")} · {fields.get("name", "")}')

    description = result.get('description', '')
    print(f'  \u00bb {description}')

    ok_flags = []
    if checks.get('extractive_triggered') is not None:
        ok = triggered == checks['extractive_triggered']
        ok_flags.append(('extraction ' + ('expected' if checks['extractive_triggered'] else 'not expected'), ok))
    if checks.get('description_non_empty'):
        ok = bool(description and description.strip())
        ok_flags.append(('description non-empty', ok))
    for term in checks.get('contains', []):
        ok = term.lower() in description.lower()
        ok_flags.append((f'contains "{term}"', ok))

    passed = all(f for _, f in ok_flags)
    for label, ok in ok_flags:
        print(f'  {"\u2705" if ok else "\u274c"} {label}')
    print()
    return passed

In [ ]:
results = []   # (case_id, passed, result_dict)
for case in CASES:
    print(f"\u2500\u2500 {case['id']}: {case['description']}")
    try:
        result = call_summarize(case['input'])
    except Exception as exc:
        print('   ERROR:', exc); print()
        results.append((case['id'], False, {}))
        continue

    passed = render(result, case)
    results.append((case['id'], passed, result))

In [ ]:
passed_n = sum(1 for _, ok, _ in results if ok)
print('SCORECARD \u00b7 /summarize  (entity_desc)')
print(f'  Cases passing: {passed_n}/{len(results)}')
for cid, ok, _ in results:
    print(f"   {'\u2705' if ok else '\u274c'} {cid}")

## ROUGE — description quality

Scores each returned `description` against the hand-written reference in the fixture.
Rows marked ⚡ used the extractive pre-reduction path.

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

case_map = {c['id']: c for c in CASES}

print(f'{"Case":<22} {"R-1":>6} {"R-2":>6} {"R-L":>6}  evidence')
print('-' * 58)
for cid, passed, result in results:
    case      = case_map[cid]
    reference = case.get('expected', {}).get('reference', '')
    desc      = result.get('description', '')
    triggered, wc = extractive_would_trigger(case['input']['fields'])

    if not reference or not desc:
        print(f'{cid:<22} {"—":>6} {"—":>6} {"—":>6}  (no ref/result)')
        continue

    scores = scorer.score(reference, desc)
    r1 = scores['rouge1'].fmeasure
    r2 = scores['rouge2'].fmeasure
    rl = scores['rougeL'].fmeasure
    ext = f'\u26a1 {wc}w' if triggered else f'  {wc}w'
    print(f'{cid:<22} {r1:>6.3f} {r2:>6.3f} {rl:>6.3f}  {ext}')